# 01 — Why static screening misses latent defects

The single claim this project rests on: **components can pass every absolute parametric limit while drifting anomalously**, and those are the ones that fail in orbit.

This notebook demonstrates it on the synthetic benchmark, where ground truth is known. Run `python scripts/generate_data.py` first.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.ingest.csv_reader import read_csv
from src.knowledge.library import MechanismLibrary
from src.simulate.lot_generator import static_screen

lib = MechanismLibrary.load()
df = read_csv(ROOT / "data" / "synthetic" / "burnin.csv", library=lib)
print(df.shape, df['part_id'].nunique(), 'parts across', df['lot_id'].nunique(), 'lots')

## 1. Static screening catches nothing

A part fails classical screening if any reading breaches its datasheet maximum at any timepoint.

In [ ]:
screen = static_screen(df)
n_defect = int(screen['is_defect'].sum())
caught = int((screen['is_defect'] & screen['static_fail']).sum())

print(f'injected latent defects : {n_defect}')
print(f'caught by static limits : {caught}')
print(f'ESCAPED                 : {n_defect - caught}')
print(f'false alarms            : {int((~screen["is_defect"] & screen["static_fail"]).sum())}')

## 2. What the defects look like

Defective trajectories drawn against the healthy population of the same lot. Note that every one stays under the dashed limit line — that is what makes them *latent*.

In [ ]:
PARAM = 'Iddq'
spec = lib.parameter(PARAM)
sub = df[df['param_name'] == PARAM]
wide = sub.pivot_table(index=['part_id', 'lot_id', 'is_defect'], columns='hours', values='value')
hours = wide.columns.to_numpy(dtype=float)

lot = wide.index.get_level_values('lot_id')[0]
block = wide.xs(lot, level='lot_id')
healthy = block.xs(False, level='is_defect')
bad = block.xs(True, level='is_defect') if True in block.index.get_level_values('is_defect') else block.iloc[:0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(hours, healthy.quantile(0.05), healthy.quantile(0.95), alpha=0.2, color='grey',
                label='healthy 5-95%')
ax.plot(hours, healthy.median(), '--', color='dimgrey', lw=2, label='lot median')
for pid, row in bad.iterrows():
    ax.plot(hours, row.to_numpy(), marker='o', lw=2, label=str(pid))
ax.axhline(spec.limit_hi, ls=':', color='firebrick', label=f'datasheet max ({spec.limit_hi} {spec.unit})')
ax.axhline(spec.derated_limit, ls='-.', color='darkorange', label='derated limit')
ax.set_xlabel('burn-in hours'); ax.set_ylabel(f'{PARAM} ({spec.unit})')
ax.set_title(f'{lot} — latent defects stay inside the limit')
ax.legend(fontsize=8); plt.tight_layout()

## 3. The signal is in the *lot-relative* value, not the absolute one

The same reading means different things in different lots. This is why Module A is lot-relative and why robust statistics (median/MAD) are used rather than mean/sigma — one extreme part inflates sigma and hides inside its own contribution to the spread.

In [ ]:
from src.features.robust_stats import mad, robust_z

at_168 = sub[sub['hours'] == 168.0].set_index('part_id')
z = robust_z(at_168['value'], at_168['lot_id'].astype(str))
is_def = at_168['is_defect'].astype(bool)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(at_168.loc[~is_def, 'value'], bins=60, alpha=0.7, label='healthy')
axes[0].hist(at_168.loc[is_def, 'value'], bins=30, alpha=0.9, label='defective')
axes[0].axvline(spec.limit_hi, ls=':', color='firebrick')
axes[0].set_title('absolute value — defects hide in the bulk'); axes[0].set_xlabel(f'{PARAM} at 168h')
axes[0].legend()

axes[1].hist(z[~is_def], bins=60, alpha=0.7, label='healthy')
axes[1].hist(z[is_def], bins=30, alpha=0.9, label='defective')
axes[1].set_title('lot-relative robust-Z — defects separate'); axes[1].set_xlabel('MAD from lot median')
axes[1].legend(); plt.tight_layout()

print(f'healthy sigma  : {at_168.loc[~is_def, "value"].std():.2f}')
print(f'with defects   : {at_168["value"].std():.2f}   <- inflated by the very parts we are hunting')
print(f'healthy MAD    : {mad(at_168.loc[~is_def, "value"].to_numpy()):.2f}')
print(f'with defects   : {mad(at_168["value"].to_numpy()):.2f}   <- barely moves')

## 4. The kinetics separate better than the level

Fitting `dV(t) = A·(t/t_ref)^n` per part gives the **Degradation Signature**. Healthy silicon sits near `n ≈ 0.18`; a runaway mechanism sits far above it — which is visible even when the absolute value is unremarkable.

In [ ]:
from src.features.build import build_features

cache = ROOT / 'data' / 'processed' / 'features.parquet'
feats = pd.read_parquet(cache) if cache.exists() else build_features(df, lib)

d = feats['is_defect'].astype(bool)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(feats.loc[~d, f'{PARAM}__v_168h'], feats.loc[~d, f'{PARAM}__n'],
           s=5, alpha=0.3, label='healthy')
ax.scatter(feats.loc[d, f'{PARAM}__v_168h'], feats.loc[d, f'{PARAM}__n'],
           s=45, color='firebrick', label='defective')
ax.axvline(spec.limit_hi, ls=':', color='firebrick')
ax.axhline(0.18, ls='--', color='grey')
ax.set_xlabel(f'{PARAM} at 168h ({spec.unit}) — the axis static screening uses')
ax.set_ylabel('fitted degradation exponent n — the axis we use')
ax.set_title('Defects are separable vertically, not horizontally')
ax.legend(); plt.tight_layout()

print(feats.groupby(d)[[f'{PARAM}__n', f'{PARAM}__identifiable', f'{PARAM}__r2']].median())

## 5. Per-mechanism view

Different mechanisms occupy different regions of signature space, which is what makes attribution possible at all — and what makes a single detector insufficient.

In [ ]:
cols = [f'{PARAM}__n_shift_vs_lot', f'{PARAM}__total_drift_frac', f'{PARAM}__jump_score',
        f'{PARAM}__noise_ratio', f'{PARAM}__rank_mobility', f'{PARAM}__r2']
summary = feats[feats['is_defect'].astype(bool)].groupby('mechanism_id')[cols].median()
summary.columns = [c.split('__')[1] for c in summary.columns]
summary.round(3)

---
## Takeaways

1. Static limits catch **0 of 100** injected latent defects with 0 false alarms. The premise holds.
2. Standard deviation is inflated by the outliers it is meant to detect; MAD is not. Hence robust statistics everywhere.
3. The absolute value at 168 h barely separates defects. The **fitted exponent** does.
4. Mechanisms occupy distinct regions of signature space — the basis for both attribution and the five-track design.

Next: `scripts/train_fusion.py`, then `scripts/run_ude.py` for the generalisation tests.